# Load COCO Reference Images and Libraries

In [1]:
!pip install -q --upgrade huggingface_hub accelerate datasets torch-fidelity torchmetrics pillow tqdm requests
!pip show torch transformers accelerate
!pip install diffusers==0.20.0 transformers==4.35.0 -q
!pip install opencv-python-headless -q
!pip install controlnet-aux -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Name: torch
Version: 2.4.1+cu124
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, torch_fidelity, torchaudio, torchmetrics, torchvision
---
Name: accelerate
Version: 1.13.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The Hugging Face team
Author-email: transformers@huggingface.co
License: Apach

In [2]:
import os
import sys

if not os.path.exists('Faster-Diffusion'):
    os.system('git clone https://github.com/hutaiHang/Faster-Diffusion.git')
else:
    print('Faster-Diffusion repo already cloned')

sys.path.insert(0, './Faster-Diffusion')

Cloning into 'Faster-Diffusion'...
Updating files: 100% (31/31), done.


In [3]:
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from torch_fidelity import calculate_metrics
from utils_sd import register_parallel_pipeline, register_faster_forward
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is depreca

In [4]:
!pip freeze > requirements.txt

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

BATCH_SIZE = 8 if DEVICE == "cuda" else 2
NUM_STEPS = 25
NUM_IMAGES = 1000   # important for stable FID

OUTPUT_ROOT = "/workspace/outputs"
COCO_DIR = "/workspace/coco_real"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(COCO_DIR, exist_ok=True)

In [6]:
import zipfile
import os

if not os.path.exists('val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip
    with zipfile.ZipFile('val2017.zip', 'r') as z:
        z.extractall('.')

if not os.path.exists('annotations'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    with zipfile.ZipFile('annotations_trainval2017.zip', 'r') as z:
        z.extractall('.')

print("COCO ready")

COCO ready


In [7]:
import zipfile

!rm -rf val2017
with zipfile.ZipFile('val2017.zip', 'r') as z:
    z.extractall('.')
print("Re-extracted val2017")

Re-extracted val2017


In [8]:
COCO_IMG_DIR = "val2017"
COCO_ANN_FILE = "annotations/captions_val2017.json"


def load_coco_val2017():
    with open(COCO_ANN_FILE, "r") as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        captions_dict.setdefault(img_id, []).append(ann["caption"])

    # Map image_id -> filename
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}

    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({
                "image_path": path,
                "caption": captions[0]  # pick first caption
            })

    return samples


samples = load_coco_val2017()
print(f"Loaded {len(samples)} samples")

Loaded 5000 samples


In [9]:
import os
from PIL import Image

OUTPUT_REAL = "/workspace/coco_real"
os.makedirs(OUTPUT_REAL, exist_ok=True)

saved = 0
skipped = 0

for i, sample in enumerate(samples):
    try:
        img = Image.open(sample["image_path"]).convert("RGB")
        img = img.resize((512, 512))
        img.save(f"{OUTPUT_REAL}/{i}.png")
        saved += 1
    except Exception as e:
        print(f"Skipping {sample['image_path']}: {e}")
        skipped += 1

print(f"Saved: {saved}, Skipped: {skipped}")

Saved: 5000, Skipped: 0


In [10]:
!df -h
!pwd
!ls /workspace/

Filesystem                         Size  Used Avail Use% Mounted on
overlay                             80G  1.1G   79G   2% /
tmpfs                               64M     0   64M   0% /dev
mfs#ca-mtl-1.runpod.net:9421       972T  739T  233T  77% /workspace
shm                                 24G  4.0K   24G   1% /dev/shm
/dev/mapper/ubuntu--vg-ubuntu--lv  455G   23G  409G   6% /usr/bin/nvidia-smi
/dev/mapper/vg-lv                  7.0T  1.4T  5.7T  20% /etc/hosts
tmpfs                              252G     0  252G   0% /sys/fs/cgroup
tmpfs                               51G  7.1M   51G   1% /run/nvidia-persistenced/socket
tmpfs                              4.0K  4.0K     0 100% /run/nvidia-ctk-hook47a1fd46-877a-435b-a7d7-a3a4edb9d84d
tmpfs                              252G     0  252G   0% /proc/acpi
tmpfs                              252G     0  252G   0% /proc/scsi
tmpfs                              252G     0  252G   0% /sys/firmware
tmpfs                              252G     0  252

### Faster-Diffusion Computation w/ FID Metric

In [11]:
def load_prompts(samples, num_images):
    return [s["caption"] for s in samples[:num_images]]


prompts = load_prompts(samples, NUM_IMAGES)

def load_pipeline(name):
    if name == "sd1.5":
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=DTYPE
        )
    elif name == "sd2.1":
        pipe = DiffusionPipeline.from_pretrained(
            "sd2-community/stable-diffusion-2-1",
            torch_dtype=DTYPE
        )

    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def enable_faster_diffusion(pipe, use_faster):
    if not use_faster:
        return
    register_parallel_pipeline(pipe, mod='50ls')
    pipe.unet.order = 0  # manually initialize order attribute
    register_faster_forward(pipe.unet, mod='50ls')

In [12]:
def generate_images(pipe, prompts, out_dir, use_faster):
    os.makedirs(out_dir, exist_ok=True)

    enable_faster_diffusion(pipe, use_faster)

    all_times = []

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size = len(batch_prompts)

        # SAME LATENTS for fairness
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE,
            dtype=DTYPE
        )

        start = time.time()

        images = pipe(
            batch_prompts,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images

        end = time.time()
        all_times.append(end - start)

        for j, img in enumerate(images):
            img.save(f"{out_dir}/{i+j}.png")

    return np.mean(all_times), np.std(all_times)

def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir,
        input2=fake_dir,
        fid=True,
        isc=False,
        kid=False,
        cuda=(DEVICE == "cuda")
    )
    return metrics["frechet_inception_distance"]

In [13]:
NUM_IMAGES = 1000
prompts = load_prompts(samples, NUM_IMAGES)

configs = [
    ("sd1.5", False),
    ("sd1.5", True),
    ("sd2.1", False),
    ("sd2.1", True),
]

results = {}

for model, faster in configs:
    print(f"\nRunning {model} | faster={faster}")

    pipe = load_pipeline(model)

    out_dir = f"{OUTPUT_ROOT}/{model}_{'faster' if faster else 'base'}"

    mean_t, std_t = generate_images(pipe, prompts, out_dir, faster)

    results[(model, faster)] = {
        "dir": out_dir,
        "time_mean": mean_t,
        "time_std": std_t
    }

    del pipe
    torch.cuda.empty_cache()


Running sd1.5 | faster=False


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.
  0%|          | 0/125 [00:00<?, ?it/s]/tmp/ipykernel_723/614101877.py:14: FutureWarning: Accessing config attribute `in_channels` directly via 'UNet2DConditionModel' object attribute is deprecated. Please access 'in_channels' over 'UNet2DConditionModel's config object instead, e.g. 'unet.config.in_channels'.
  (batch_size, pipe.unet.in_channels, 64, 64),
100%|██████████| 125/125 [16:06<00:00,  7.74s/it]



Running sd1.5 | faster=True


...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 125/125 [13:51<00:00,  6.65s/it]



Running sd2.1 | faster=True


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 125/125 [14:02<00:00,  6.74s/it]


In [14]:
fid_results = {}

for model in ["sd1.5", "sd2.1"]:
    base_dir = results[(model, False)]["dir"]
    faster_dir = results[(model, True)]["dir"]

    fid_base = compute_fid(COCO_DIR, base_dir)
    fid_faster = compute_fid(COCO_DIR, faster_dir)

    fid_results[model] = {
        "base": fid_base,
        "faster": fid_faster,
        "delta": fid_faster - fid_base
    }

print("\n=== FINAL RESULTS ===")

for (model, faster), data in results.items():
    print(f"{model} | faster={faster}")
    print(f"  Time: {data['time_mean']:.3f} \u00b1 {data['time_std']:.3f}")

for model, vals in fid_results.items():
    print(f"\n{model} FID:")
    print(f"  Base:   {vals['base']:.3f}")
    print(f"  Faster: {vals['faster']:.3f}")
    print(f"  \u0394FID:   {vals['delta']:.3f}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 146MB/s]
Extracting statistics from input 1
Looking for samples non-recursivelty in "/workspace/coco_real" with extensions png,jpg,jpeg
Found 5000 samples
Processing samples                                                           
Extracting statistics from input 2
Looking for samples non-recursivelty in "/workspace/outputs/sd1.5_base" with extensions png,jpg,jpeg
Found 1000 samples
Processing samples                                                           
Frechet Inception Distance: 53.59898
Creating feature extractor "inception-v3-compat" with features ['2048']
Extracting statistics from input 1
Looking for samples non-recursivelty in "/workspace/coco_real" with exte


=== FINAL RESULTS ===
sd1.5 | faster=False
  Time: 6.925 ± 0.071
sd1.5 | faster=True
  Time: 7.021 ± 0.068
sd2.1 | faster=False
  Time: 5.823 ± 0.028
sd2.1 | faster=True
  Time: 5.915 ± 0.024

sd1.5 FID:
  Base:   53.599
  Faster: 55.187
  ΔFID:   1.588

sd2.1 FID:
  Base:   58.546
  Faster: 59.019
  ΔFID:   0.473


Frechet Inception Distance: 59.01874


### CLIP Computation

In [15]:
def load_clip():
    model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32"
    ).to(DEVICE)

    processor = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32"
    )

    model.eval()
    return model, processor

In [16]:
def compute_clip_score(image_dir, prompts, model, processor):
    image_files = sorted(os.listdir(image_dir))

    scores = []

    for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
        batch_files = image_files[i:i+BATCH_SIZE]
        batch_images = []
        batch_texts = []

        for j, file in enumerate(batch_files):
            img = Image.open(os.path.join(image_dir, file)).convert("RGB")
            batch_images.append(img)

            # match prompt index
            idx = i + j
            batch_texts.append(prompts[idx])

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            # Normalize (CRITICAL)
            image_embeds = F.normalize(image_embeds, dim=-1)
            text_embeds = F.normalize(text_embeds, dim=-1)

            # cosine similarity
            batch_scores = (image_embeds * text_embeds).sum(dim=-1)

        scores.extend(batch_scores.cpu().numpy())

    return float(np.mean(scores))

In [17]:
clip_model, clip_processor = load_clip()

clip_results = {}

for model in ["sd1.5", "sd2.1"]:
    base_dir = results[(model, False)]["dir"]
    faster_dir = results[(model, True)]["dir"]

    print(f"\nComputing CLIP for {model}...")

    clip_base = compute_clip_score(
        base_dir, prompts, clip_model, clip_processor
    )

    clip_faster = compute_clip_score(
        faster_dir, prompts, clip_model, clip_processor
    )

    clip_results[model] = {
        "base": clip_base,
        "faster": clip_faster,
        "delta": clip_faster - clip_base
    }

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_file, map_location=map


Computing CLIP for sd1.5...


100%|██████████| 125/125 [01:17<00:00,  1.61it/s]



Computing CLIP for sd2.1...


100%|██████████| 125/125 [01:21<00:00,  1.53it/s]


# Results

In [18]:
for model, vals in clip_results.items():
    print(f"\n{model} CLIP:")
    print(f"  Base:   {vals['base']:.4f}")
    print(f"  Faster: {vals['faster']:.4f}")
    print(f"  \u0394CLIP:  {vals['delta']:.4f}")


sd1.5 CLIP:
  Base:   0.1459
  Faster: 0.1453
  ΔCLIP:  -0.0006

sd2.1 CLIP:
  Base:   0.1513
  Faster: 0.1532
  ΔCLIP:  0.0019


In [19]:
# Print full timing + metric summary matching DeepCache output format
print("\n=== FULL SUMMARY ===")
for (model, faster), data in results.items():
    print(f"{model} | faster={faster}")
    print(f"  Time: {data['time_mean']:.3f} \u00b1 {data['time_std']:.3f}")

for model in ["sd1.5", "sd2.1"]:
    print(f"\n{model} FID:")
    print(f"  Base:   {fid_results[model]['base']:.3f}")
    print(f"  Faster: {fid_results[model]['faster']:.3f}")
    print(f"  \u0394FID:   {fid_results[model]['delta']:.3f}")
    print(f"\n{model} CLIP:")
    print(f"  Base:   {clip_results[model]['base']:.4f}")
    print(f"  Faster: {clip_results[model]['faster']:.4f}")
    print(f"  \u0394CLIP:  {clip_results[model]['delta']:.4f}")


=== FULL SUMMARY ===
sd1.5 | faster=False
  Time: 6.925 ± 0.071
sd1.5 | faster=True
  Time: 7.021 ± 0.068
sd2.1 | faster=False
  Time: 5.823 ± 0.028
sd2.1 | faster=True
  Time: 5.915 ± 0.024

sd1.5 FID:
  Base:   53.599
  Faster: 55.187
  ΔFID:   1.588

sd1.5 CLIP:
  Base:   0.1459
  Faster: 0.1453
  ΔCLIP:  -0.0006

sd2.1 FID:
  Base:   58.546
  Faster: 59.019
  ΔFID:   0.473

sd2.1 CLIP:
  Base:   0.1513
  Faster: 0.1532
  ΔCLIP:  0.0019


In [20]:
| Model | Speedup | FID ↑ | CLIP Δ |
| ----- | ------- | ----- | ------ |
| SD1.5 | 0.99×   | 1.588 | -0.0006 |
| SD2.1 | 0.98×   | 0.473 | +0.0019 |

| Faster-Diffusion Model Name     | Mean Time  | Var. Time  | FID Score | CLIP Score |
| ------------------------------- | ---------- | ---------- | --------- | ---------- |
| SD1.5, Faster=False             | 6.925 secs | 0.071 secs | 53.599    | 0.1459     |
| SD1.5, Faster=True  (mod=50ls)  | 7.021 secs | 0.068 secs | 55.187    | 0.1453     |
| SD2.1, Faster=False             | 5.823 secs | 0.028 secs | 58.546    | 0.1513     |
| SD2.1, Faster=True  (mod=50ls)  | 5.915 secs | 0.024 secs | 59.019    | 0.1532     |

SyntaxError: invalid character '↑' (U+2191) (2207185421.py, line 1)